# Explore Robot Segmentation (Steps 1–3 + Stabilized Meta SAM 3.1)

Standalone exploration of the AL5D `random-both-cameras-video` demopack:
1. **Step 1**: Episode inventory, moving vs. paused frame timelines, and frozen 6/2/2 split.
2. **Step 2**: Per-camera median background, MAD variation, and static positional control.
3. **Step 3**: Offline teachers (`motion` vs. **Meta SAM 3.1**), including **Stages A–C controller-board stabilization** (occlusion-safe static `"circuit board"` inclusion as robot FG, whole-assembly hypothesis consistency, and static-region consensus `UNKNOWN=255` gating).

In [ ]:
pack_path = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Demopacks/random-both-cameras-video"
result_dir = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/RobotSegmentation_pilot"
camera = "dev2"
recompute = False
pilot_episodes = None
max_frames = 150
run_sam3 = True
sam3_checkpoint = "/Users/lboloni/Documents/Develop/Data/BerryPicker-data-external/sam3/sam3.1_multiplex.pt"
sam3_prompt = "robot arm"
sam3_pcb_prompts = ["circuit board"]
sam3_pcb_conf = 0.50
sam3_selection_mode = "union"
sam3_conf = 0.25
sam3_imgsz = 644
sam3_step = 4
recompute_sam3 = False
device = "auto"
seed = 0

In [ ]:
import json
import sys
from pathlib import Path

src_root = Path.cwd().resolve()
while src_root.name != "src" and src_root.parent != src_root:
    if (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
        break
    src_root = src_root.parent
if not (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
    src_root = Path("/Users/lboloni/Documents/Develop/Github/BerryPicker/BerryPicker/src")
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from _experimental._2026_11_robot_segmentation.baselines import MedianBackgroundModel, StaticMaskControl
from _experimental._2026_11_robot_segmentation.data import (
    BG, FG, UNKNOWN,
    camera_video_path, detect_pause_flags, frame_motion_scores,
    inventory_demopack, load_label_png, load_split, make_episode_split,
    read_video_frames, render_mask_overlay, save_split,
)
from _experimental._2026_11_robot_segmentation.evaluate import extract_annotation_pack
from _experimental._2026_11_robot_segmentation.teachers import (
    MotionLabelTeacher, SAM3DirectSegmenter, SAM3Teacher,
    apply_static_consensus_gating, estimate_static_pcb_mask, save_teacher_outputs,
)
from _experimental._2026_11_robot_segmentation.viz import make_comparison_strip, plot_episode_inventory

out = Path(result_dir)
out.mkdir(parents=True, exist_ok=True)

def show_rgb(rgb):
    bgr = cv2.cvtColor(np.asarray(rgb, dtype=np.uint8), cv2.COLOR_RGB2BGR)
    display(Image(data=cv2.imencode(".png", bgr)[1].tobytes()))

print("pack_path      :", pack_path)
print("result_dir     :", out)
print("camera         :", camera)
print("sam3_checkpoint:", sam3_checkpoint, "(exists=" + str(Path(sam3_checkpoint).is_file()) + ")")

## Step 1 — Episode Inventory & Frozen 6/2/2 Split

In [ ]:
inv_path = out / "inventory.json"
split_path = out / "split.json"
if recompute or not inv_path.is_file():
    inv = inventory_demopack(pack_path, cameras=("dev2", "dev3"), eval_frames_per_episode=12, seed=seed)
    inv_path.write_text(json.dumps(inv, indent=2) + "\n", encoding="utf-8")
    split = make_episode_split(pack_path, camera="dev2")
    save_split(split, split_path)
else:
    inv = json.loads(inv_path.read_text(encoding="utf-8"))
    split = load_split(split_path)
eval_pack_dir = out / f"eval_pack_{camera}"
if recompute or not (eval_pack_dir / "eval_manifest.json").is_file():
    extract_annotation_pack(inv, eval_pack_dir, camera=camera)

print("Train (6):", split.train)
print("Val   (2):", split.val)
print("Test  (2):", split.test, "(kept sealed until Step 7)")

fig = plot_episode_inventory(inv, camera=camera)
fig.savefig(out / f"inventory_{camera}.png", dpi=130)
plt.show()

## Step 2 — Median-Background Baseline & Static Positional Control

In [ ]:
bg_dir = out / "median_background" / camera
static_dir = out / "static_control" / camera

if recompute or not (bg_dir / "config.json").is_file():
    bg = MedianBackgroundModel.fit_from_demopack(pack_path, split.train, camera=camera, frame_step=10, threshold=0.12, dilate=1)
    bg.save(bg_dir)
    train_masks = []
    for ep in split.train:
        _, frms = read_video_frames(camera_video_path(pack_path, ep, camera), step=10)
        train_masks.extend(bg.segment(f) for f in frms)
    static = StaticMaskControl.fit_from_masks(train_masks, threshold=0.5, camera_id=camera)
    static.save(static_dir)
else:
    bg = MedianBackgroundModel.load(bg_dir)
    static = StaticMaskControl.load(static_dir)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
axes[0].imshow(bg.background_rgb)
axes[0].set_title(f"Median background ({camera})")
axes[1].imshow(bg.mad_rgb.max(axis=2), cmap="inferno")
axes[1].set_title("Per-pixel MAD (variation)")
axes[2].imshow(static.occupancy, cmap="viridis", vmin=0, vmax=1)
axes[2].set_title("Static occupancy (train)")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(out / f"baselines_{camera}.png", dpi=130)
plt.show()

## Step 3 — Offline Teachers: Motion vs. Stabilized Meta SAM 3.1 (Stages A–C)

Stage A detects the attached controller PCB once on the median background (`"circuit board"`) and includes it in the robot mask (`FG=1`) wherever the current frame matches the background (so unoccluded PCB pixels are always FG while non-robot occluders are not forced into FG). Stage B keeps the PCB and high-consensus static base (`p >= 0.85`) as `FG (1)` and marks remaining intermediate-frequency static edges (`0.15 < p < 0.85`) as `UNKNOWN (255)`.

In [ ]:
episodes_to_label = list(pilot_episodes) if pilot_episodes is not None else (split.train + split.val)
motion_teacher = MotionLabelTeacher(bg)
motion_summaries = {}
motion_results = {}

for ep in episodes_to_label:
    manifest_path = out / "teachers" / "motion" / camera / ep / "manifest.json"
    if recompute or not manifest_path.is_file():
        ids, frms = read_video_frames(camera_video_path(pack_path, ep, camera), max_frames=max_frames)
        res = motion_teacher.label_episode(frms, ids, episode=ep, camera=camera)
        s = save_teacher_outputs(res, frms, out / "teachers", save_overlays_every=25)
        motion_results[ep] = res
    else:
        s = json.loads(manifest_path.read_text(encoding="utf-8"))
    motion_summaries[ep] = s

sam3_raw_seg = None
sam3_stab_seg = None
pcb_static = None
static_freq = None
sam3_summaries = {}

if run_sam3 and Path(sam3_checkpoint).is_file():
    sam3_raw_seg = SAM3DirectSegmenter(
        checkpoint=sam3_checkpoint, prompt=sam3_prompt, conf=sam3_conf, imgsz=sam3_imgsz,
        device=device, camera_id=camera, selection_mode="union",
    )
    sam3_raw_seg.save(out / "models" / f"sam3_raw_{camera}")

    # Stage A: static PCB mask on median background (included as robot FG)
    consensus_npz = out / "teachers" / f"sam3_static_consensus_{camera}.npz"
    pcb_static = estimate_static_pcb_mask(
        sam3_raw_seg, bg, pcb_prompts=tuple(sam3_pcb_prompts), conf=sam3_pcb_conf, dilate_px=2
    )
    print(f"Stage A static PCB included ({sam3_pcb_prompts}): {int(pcb_static.sum())} px on {camera}")

    # Stabilized segmenter (Stages A + C: PCB included as part of robot)
    sam3_stab_seg = SAM3DirectSegmenter(
        checkpoint=sam3_checkpoint, prompt=sam3_prompt, conf=sam3_conf, imgsz=sam3_imgsz,
        device=device, camera_id=camera, selection_mode=sam3_selection_mode,
        bg_model=bg, pcb_mask=pcb_static, bg_match_threshold=0.10,
    )
    sam3_stab_seg.save(out / "models" / f"sam3_direct_{camera}")

    need_sam3 = recompute or recompute_sam3 or not consensus_npz.is_file() or any(
        not (out / "teachers" / "sam3" / camera / ep / "manifest.json").is_file() for ep in episodes_to_label
    )
    if need_sam3:
        sam3_teacher = SAM3Teacher(
            prompt=sam3_prompt, checkpoint=sam3_checkpoint, conf=sam3_conf, imgsz=sam3_imgsz,
            device=device, segmenter=sam3_stab_seg,
        )
        res_by_ep = {}
        frms_by_ep = {}
        for ep in episodes_to_label:
            ids, frms = read_video_frames(
                camera_video_path(pack_path, ep, camera),
                step=sam3_step,
                max_frames=(None if max_frames is None else max(1, max_frames // sam3_step)),
            )
            res_by_ep[ep] = sam3_teacher.label_episode(frms, ids, motion_results.get(ep), episode=ep, camera=camera)
            frms_by_ep[ep] = frms

        # Stage B: static consensus gating across training episodes (PCB included as FG)
        tr_only = {ep: res_by_ep[ep] for ep in episodes_to_label if ep in split.train}
        tr_frms = {ep: frms_by_ep[ep] for ep in tr_only}
        gated_tr, static_freq = apply_static_consensus_gating(
            tr_only, tr_frms, bg, low_freq=0.15, high_freq=0.85, pcb_mask=pcb_static
        )
        res_by_ep.update(gated_tr)
        np.savez_compressed(consensus_npz, static_freq=static_freq, pcb_mask=pcb_static)
        for ep in episodes_to_label:
            sam3_summaries[ep] = save_teacher_outputs(res_by_ep[ep], frms_by_ep[ep], out / "teachers", save_overlays_every=10)
    else:
        arrs = np.load(consensus_npz)
        static_freq = arrs["static_freq"]
        for ep in episodes_to_label:
            sam3_summaries[ep] = json.loads((out / "teachers" / "sam3" / camera / ep / "manifest.json").read_text(encoding="utf-8"))

rows = ["| Teacher | Episode | Split | Total | Retained | Moving retained | Pause retained |", "|---|---|---|---:|---:|---:|---:|"]
for tname, smap in [("motion", motion_summaries), ("sam3 (3.1 stab)", sam3_summaries)]:
    for ep, s in smap.items():
        part = "train" if ep in split.train else "val"
        rows.append(
            f"| {tname} | {ep} | {part} | {s['total_frames']} | "
            f"{s['retained_frames']} ({100*s['retained_fraction']:.1f}%) | "
            f"{s['moving_retained']}/{s['moving_total']} | "
            f"{s['pause_retained']}/{s['pause_total']} |"
        )
display(Markdown("\n".join(rows)))

### Controller-Board Stabilization Diagnostics (Stage A Distractor Mask & Stage B Static Consensus)

In [ ]:
if pcb_static is not None and static_freq is not None:
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.6))
    axes[0].imshow(render_mask_overlay(bg.background_rgb, pcb_static))
    axes[0].set_title(f"Stage A: static PCB included ({int(pcb_static.sum())} px)")
    im = axes[1].imshow(static_freq, cmap="coolwarm", vmin=0, vmax=1)
    axes[1].set_title("Stage B: static FG frequency p(x,y)")
    fig.colorbar(im, ax=axes[1], fraction=0.046)
    consensus_map = np.full(pcb_static.shape, BG, dtype=np.uint8)
    valid = np.isfinite(static_freq)
    consensus_map[valid & (static_freq >= 0.85)] = FG
    consensus_map[valid & (static_freq > 0.15) & (static_freq < 0.85)] = UNKNOWN
    consensus_map[pcb_static > 0] = FG
    axes[2].imshow(render_mask_overlay(bg.background_rgb, consensus_map))
    axes[2].set_title("Stage B: static FG(green) / UNK(amber)")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(out / f"sam3_stabilization_{camera}.png", dpi=130)
    plt.show()

### Validation Frames: Raw SAM 3.1 vs. Stabilized SAM 3.1 (Moving & Paused)

In [ ]:
val_ep = next(ep for ep in episodes_to_label if ep in split.val)
ids, frms = read_video_frames(camera_video_path(pack_path, val_ep, camera), max_frames=max_frames)
motion = frame_motion_scores(frms)
pauses, _ = detect_pause_flags(motion)

move_idx = np.flatnonzero(~pauses)
pause_idx = np.flatnonzero(pauses)
picks = []
if len(move_idx):
    picks.extend(move_idx[np.linspace(0, len(move_idx) - 1, min(3, len(move_idx))).astype(int)].tolist())
if len(pause_idx):
    picks.extend(pause_idx[np.linspace(0, len(pause_idx) - 1, min(2, len(pause_idx))).astype(int)].tolist())

m_lbl_dir = out / "teachers" / "motion" / camera / val_ep / "training_labels"
if sam3_stab_seg is not None:
    sam3_stab_seg.reset_state()
for idx in picks:
    fidx = ids[idx]
    rgb = frms[idx]
    panels = {
        "Median-BG": bg.segment(rgb),
        "Motion": load_label_png(m_lbl_dir / f"frame_{fidx:06d}.png"),
    }
    if sam3_raw_seg is not None:
        panels["SAM 3.1 Raw"] = sam3_raw_seg.segment(rgb)
    if sam3_stab_seg is not None:
        panels["SAM 3.1 Stab"] = sam3_stab_seg.segment(rgb)
    badge = f"f{fidx} {'PAUSE' if pauses[idx] else 'MOVE'}"
    show_rgb(make_comparison_strip(rgb, panels, badge=badge))